In [ ]:
# imports
import pandas as pd
import numpy as np
import requests
import time
from bs4 import BeautifulSoup
import re
from selenium import webdriver
import random

In [ ]:
# Új dataframe létrehozása
if input("Ha tényleg új dataframe-t szeretnél, írd le hogy 'Igen'") == "Igen":
    cities = {
        "Barcelona": {"country": "Spain"},
        "Lisbon": {"country": "Portugal"},
        "Tirana": {"country": "Albania"},
        "Geneva": {"country": "Switzerland"},
        "Antalya": {"country": "Turkey"},
        "Amsterdam": {"country": "Netherlands"},
        "Berlin": {"country": "Germany"},
        "Paris": {"country": "France"},
        "Rome": {"country": "Italy"},
        "Madrid": {"country": "Spain"},
        "Athens": {"country": "Greece"},
        "Vienna": {"country": "Austria"},
        "London": {"country": "United Kingdom"},
        "Warsaw": {"country": "Poland"},
        "Brussels": {"country": "Belgium"},
        "Prague": {"country": "Czech Republic"},
        "Milan": {"country": "Italy"},
        "Copenhagen": {"country": "Denmark"},
        "Dubrovnik": {"country": "Croatia"},
        "Oslo": {"country": "Norway"}
    }

    df = pd.DataFrame(cities).T
    print("Új dataframe:")
    display(df.sample(5))
else:
    print("Új dataframe készítése megszakítva.")

In [ ]:
# Földrajz API

# Fő földrajzi típusok Overpass kulcsszavai
geo_types = {
    "beach": "natural=beach",
    "mountain": "natural=peak",
    "lake": "natural=lake",
    "desert": "natural=desert",
    "island": "place=island",
    "attraction": "tourism=attraction",
    "park": "leisure=park", 
    "monument": "historic=monument",
}

def get_coordinates(city, country):
    """
    Visszaadja a (lat, lon) koordinátákat egy város és ország alapján
    """
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "city": city,
        "country": country,
        "format": "json",
        "limit": 1
    }
    
    response = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
    
    if response.status_code == 200 and response.json():
        data = response.json()[0]
        lat = float(data["lat"])
        lon = float(data["lon"])
        return lat, lon
    else:
        return None, None
    
def get_geo_scores(lat, lon, radius=10000):
    """
    Lekérdezi az Overpass API-t, és visszaadja a fő földrajzi típusokra
    a találatok számát normalizált 0-1 skálán.
    """
    scores = {}
    
    for typ, tag in geo_types.items():
        query = f"""
        [out:json][timeout:25];
        (
          node["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
          way["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
          relation["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
        );
        out center;
        """
        url = "http://overpass-api.de/api/interpreter"
        response = requests.post(url, data={"data": query})

        time.sleep(random.uniform(0.3,0.6))  # rate limit elkerülése
        
        if response.status_code == 200:
            data = response.json()
            count = len(data["elements"])
            scores[typ] = count
        else:
            scores[typ] = -1  # hiba esetén -1
    
    return scores

# Először mentsük el a koordinátákat
for city, row in df.iterrows():
    lat, lon = get_coordinates(city, row["country"])
    df.loc[city, "lat"] = lat
    df.loc[city, "lon"] = lon

for city, row in df.iterrows():
    lat, lon = df.loc[city, ["lat", "lon"]]
    scores = get_geo_scores(lat, lon)
    print(f"\n{city} földrajzi típus pontszámok:")
    for k, v in scores.items():
        df.loc[city, f"geo_{k}"] = v
        print(f"  {k}: {v:.2f}")


In [ ]:
# Költségek (numbeo)

def clean_col_name(name):
    """
    Tisztítja az oszlopneveket:
    - kisbetűs
    - minden speciális karaktert aláhúzásra cserél
    - többszörös aláhúzásból egyet csinál
    """
    name = name.lower()
    # Cseréljük a nem alfanumerikus karaktereket aláhúzásra
    name = re.sub(r'[^a-z0-9]+', '_', name)
    # Többszörös aláhúzás → 1 aláhúzás
    name = re.sub(r'_+', '_', name)
    # Elejéről és végéről aláhúzás eltávolítása
    name = name.strip('_')
    return name


for city, row in df.iterrows():
    # URL encode a városnévhez, ha van szóköz
    city_url = city.replace(" ", "-")
    url = f"https://www.numbeo.com/cost-of-living/in/{city_url}?displayCurrency=EUR"

    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("table", {"class": "data_wide_table"})
    if table is None:
        print("  Nincs adat a városhoz.")
        continue

    rows = table.find_all("tr")
    for tr in rows:
        cols = tr.find_all("td")
        if len(cols) >= 2:
            item = cols[0].text.strip()
            value = cols[1].text.strip()
            try:
                value_num = float(value.replace("€","").replace(",","").strip())
            except:
                value_num = None

            # Tisztított oszlopnév
            col_name = "col_" + clean_col_name(item)
            df.loc[city, col_name] = value_num


In [ ]:
# Klíma

def get_hourly_weather(latitude, longitude, start_date, end_date):
    # pip install openmeteo-requests
    # pip install requests_cache
    # pip install retry-requests
    
    import openmeteo_requests
    import requests_cache
    import pandas as pd
    from retry_requests import retry
    
    # Open-Meteo API kliens beállítása gyorsítótárral és hibakezeléssel
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    # API hívás paramétereinek beállítása
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,  # szélességi fok
        "longitude": longitude,  # hosszúsági fok
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,wind_speed_10m,weathercode"
    }
    
    # API hívás végrehajtása
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    
    # Óránkénti adatok feldolgozása
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(1).ValuesAsNumpy()
    hourly_weathercode = hourly.Variables(2).ValuesAsNumpy()
    
    # Időbélyegek létrehozása
    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=False),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=False),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature_2m": hourly_temperature_2m,
        "wind_speed_10m": hourly_wind_speed_10m,
        "weathercode": hourly_weathercode
    }
    
    # DataFrame létrehozása
    hourly_dataframe = pd.DataFrame(data=hourly_data)
    
    # Időjárás kódok értelmezése
    weather_descriptions = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Intense drizzle",
        56: "Light freezing drizzle",
        57: "Intense freezing drizzle",
        61: "Light rain",
        63: "Moderate rain",
        65: "Heavy rain",
        66: "Light freezing rain",
        67: "Heavy freezing rain",
        71: "Light snow",
        73: "Moderate snow",
        75: "Heavy snow",
        77: "Hail",
        80: "Light showers",
        81: "Moderate showers",
        82: "Heavy showers",
        85: "Light snow showers",
        86: "Heavy snow showers",
        95: "Light or moderate thunderstorm",
        96: "Light thunderstorm with hail",
        99: "Severe thunderstorm with hail"
    }
    
    # Az időjárás kódok leírásának hozzáadása a DataFrame-hez
    hourly_dataframe["weather_description"] = hourly_dataframe["weathercode"].map(weather_descriptions)
    # Átváltás m/s-ról km/h-ra
    hourly_dataframe["wind_speed_kmh"] = hourly_dataframe["wind_speed_10m"] * 3.6
    
    # Eredmény kiíratása
    output_df = hourly_dataframe[["date", "temperature_2m", "wind_speed_kmh", "weather_description", "weathercode"]]
    output_df = output_df.copy()
    output_df.rename(columns={"temperature_2m": "temp_celsius"}, inplace=True)
        
    return output_df

for city, row in df.iterrows():
    lat, lon = df.loc[city, ["lat", "lon"]]
    if lat is None:
        continue

    # archive API (példa 2023-as év)
    climate_df = get_hourly_weather(lat, lon, "2023-01-01", "2023-12-31")

    # Egyszerű havi aggregáció (átlag hőmérséklet)
    climate_df['month'] = climate_df['date'].dt.month
    monthly_avg = climate_df.groupby('month')['temp_celsius'].mean()

    for month, value in monthly_avg.items():
        col_name = f"climate_temp_mean_{month}"
        df.loc[city, col_name] = value

print(df[[col for col in df.columns if 'climate_temp' in col][:3]].head(5))


In [ ]:
# Életstílus

# Dimenziók és OSM kulcs-érték párok
lifestyle_categories = {
    "bulis": [
        {"key": "amenity", "value": "bar"},
        {"key": "amenity", "value": "pub"},
        {"key": "amenity", "value": "nightclub"},
    ],
    "relax": [
        {"key": "leisure", "value": "park"},
        {"key": "natural", "value": "beach"},
        {"key": "leisure", "value": "garden"},
    ],
    "kulturalis": [
        {"key": "tourism", "value": "museum"},
        {"key": "tourism", "value": "artwork"},
        {"key": "historic", "value": "monument"},
    ],
    "csaladbarat": [
        {"key": "amenity", "value": "kindergarten"},
        {"key": "leisure", "value": "playground"},
        {"key": "tourism", "value": "zoo"},
    ]
}

import time

def query_overpass(lat, lon, key, value, radius=10000):
    overpass_url = "http://overpass-api.de/api/interpreter"
    query = f"""
    [out:json][timeout:25];
    (
      node["{key}"="{value}"](around:{radius},{lat},{lon});
      way["{key}"="{value}"](around:{radius},{lat},{lon});
      relation["{key}"="{value}"](around:{radius},{lat},{lon});
    );
    out center;
    """
    response = requests.post(overpass_url, data={"data": query})
    if response.status_code == 200:
        data = response.json()
        return len(data.get("elements", []))
    else:
        return 0

def get_lifestyle_scores(city, country, radius=10000):
    lat, lon = get_coordinates(city, country)
    if not lat or not lon:
        return None

    scores = {}
    for dimension, cat_list in lifestyle_categories.items():
        count = 0
        for cat in cat_list:
            count += query_overpass(lat, lon, cat["key"], cat["value"], radius)
            time.sleep(1)  # rate limit miatt
        # egyszerű normalizálás: max 0-1 (tetszőleges normalizációs logika később)
        scores[dimension] = count
    return scores

for city, row in df.iterrows():
    scores = get_lifestyle_scores(city, row['country'], radius=10000)
    print(f"\n{city} életstílus pontszámok:")
    for dim, val in scores.items():
        df.loc[city, f"eletstilus_{dim}"] = val
        print(f"  {dim}: {val}")


In [ ]:
# Távolság
 
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

start_city, start_country = "Budapest", "Hungary"
lat_start, lon_start = get_coordinates(start_city, start_country)

for city, row in df.iterrows():
    lat, lon = df.loc[city, ["lat", "lon"]]
    dist = haversine(lat_start, lon_start, lat, lon)

    df.loc[city, 'distance'] = dist
    print(f"{start_city}-{city} távolság:", round(dist, 1), "km")


In [ ]:
# Turista sűrűség

# Példa POI-sűrűség számítására
def get_city_crowding_score(lat, lon, radius=10000):
    categories = ["tourism=museum", "tourism=attraction", "amenity=hotel"]
    total_count = 0
    for cat in categories:
        key, value = cat.split("=")
        total_count += query_overpass(lat, lon, key, value, radius)
    return total_count

for city, row in df.iterrows():
    lat, lon = df.loc[city, ["lat", "lon"]]
    print(f"{city}: {get_city_crowding_score(lat, lon)}")

In [ ]:
# Population

from SPARQLWrapper import SPARQLWrapper, JSON

def get_wikidata_population_by_coord(lat, lon, radius_km=10):
    """
    Koordináta alapján lekéri a legközelebbi város népességét Wikidatából.
    radius_km: környező keresési sugár
    """
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

    # Koordináta-keresés Wikidata-n (geóval)
    query = f"""
    SELECT ?city ?cityLabel ?population ?coord WHERE {{
      ?city wdt:P31/wdt:P279* wd:Q515;
            wdt:P1082 ?population;
            wdt:P625 ?coord.
      SERVICE wikibase:around {{
        ?city wdt:P625 ?location .
        bd:serviceParam wikibase:center "Point({lon} {lat})"^^geo:wktLiteral .
        bd:serviceParam wikibase:radius "{radius_km}" .
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY DESC(?population)
    LIMIT 1
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    time.sleep(3.5)

    if results["results"]["bindings"]:
        pop = int(results["results"]["bindings"][0]["population"]["value"])
        label = results["results"]["bindings"][0]["cityLabel"]["value"]
        return label, pop
    else:
        return None, None

for city, row in df.iterrows():
    lat, lon = df.loc[city, ["lat", "lon"]]
    label, pop = get_wikidata_population_by_coord(lat, lon)
    df.loc[city, "population"] = pop
    print(f"{city} -> {label}: {pop}")


In [ ]:
# Értékek normalizálása

# Csak az eredeti oszlopok normalizálása (ne az újonnan létrehozottaké)
geo_cols = [col for col in df.columns if col.startswith('geo_') and '_per_100k' not in col]
lifestyle_cols = [col for col in df.columns if col.startswith('eletstilus_') and '_per_100k' not in col]

for city, row in df.iterrows():
    if pd.notna(row.get('population')) and row['population'] > 0:
        # Földrajzi értékek normalizálása
        for geo_col in geo_cols:
            df.loc[city, f'{geo_col}_per_100k'] = (df.loc[city, geo_col] / row['population']) * 100000
        
        # Életstílus értékek normalizálása
        for lifestyle_col in lifestyle_cols:
            df.loc[city, f'{lifestyle_col}_per_100k'] = (df.loc[city, lifestyle_col] / row['population']) * 100000
        
        # Turista sűrűség normalizálása
        crowding_score = get_city_crowding_score(row['lat'], row['lon'])
        df.loc[city, 'crowding_per_100k'] = (crowding_score / row['population']) * 100000

In [ ]:
df.to_excel('cities20.xlsx', index=True)